# 08 — Movie Quality Labels

Assigns a quality label to each movie in `data/tmdb_movies_clean.csv`.

## Why not a null-score threshold?

A simple count of missing fields treats all signals as equal weight — but
`vote_count > 0` is a much stronger indicator than `has_tagline`. And some
legitimate films have no IMDB ID (*Once Upon a Deadpool*, *Hulk vs. Wolverine*)
while some spam entries have overviews that are apartment rental listings.

## Three dimensions

| Dimension | Signal |
|-----------|--------|
| **Real vs spam** | human engagement (votes, IMDB cross-ref, plausible overview) |
| **Theatrical vs video** | `video=False` in TMDB daily export |
| **Fresh vs stale** | Kaggle snapshot may lag live API by months |

## Label scheme

| Label | Meaning |
|-------|---------|
| `real_confident` | IMDB ID + vote_count > 5 — almost certainly a real film |
| `real_likely` | Has human engagement signal |
| `stub_legitimate` | Upcoming/obscure — real but no audience yet |
| `stub_uncertain` | Insufficient signal — could go either way |
| `spam_likely` | No signal at all |
| `spam_confident` | Completely empty shell |

Rules applied in order — first match wins.

In [1]:
import pandas as pd

df   = pd.read_csv('data/tmdb_movies_clean.csv', low_memory=False)
tmdb = pd.read_csv('data/tmdb_movie_ids.csv')[['id', 'video']]

# Join video flag (theatrical = video is False)
df = df.merge(tmdb, on='id', how='left')
df['theatrical'] = ~df['video'].fillna(False)  # True = theatrical / False = video release

print(f'Loaded {len(df):,} movies')
print(f'Theatrical (video=False): {df["theatrical"].sum():,}')
print(f'Video release:            {(~df["theatrical"]).sum():,}')

Loaded 1,169,756 movies
Theatrical (video=False): 1,111,990
Video release:            57,766


## Signals

In [2]:
has_imdb      = df['imdb_id'].notna()
has_overview  = df['overview'].notna() & (df['overview'].str.strip() != '')
has_genres    = df['genres'].notna()
has_release   = df['release_date'].notna()
has_runtime   = df['runtime'].fillna(0) > 0
has_companies = df['production_companies'].notna()
theatrical    = df['theatrical']
votes         = df['vote_count'].fillna(0)

coverage = {
    'has_imdb':      has_imdb.sum(),
    'has_overview':  has_overview.sum(),
    'has_genres':    has_genres.sum(),
    'has_release':   has_release.sum(),
    'has_runtime':   has_runtime.sum(),
    'has_companies': has_companies.sum(),
    'theatrical':    theatrical.sum(),
    'votes > 0':     (votes > 0).sum(),
    'votes > 5':     (votes > 5).sum(),
    'votes > 20':    (votes > 20).sum(),
}
for k, v in coverage.items():
    print(f'{k:<20} {v:>9,}  ({v/len(df)*100:.1f}%)')

has_imdb               615,529  (52.6%)
has_overview           914,826  (78.2%)
has_genres             736,647  (63.0%)
has_release            935,033  (79.9%)
has_runtime            822,735  (70.3%)
has_companies          470,896  (40.3%)
theatrical           1,111,990  (95.1%)
votes > 0              330,706  (28.3%)
votes > 5              109,916  (9.4%)
votes > 20              48,720  (4.2%)


## Apply quality labels

In [3]:
# Rules applied in order — first match wins
# Adjust thresholds here if inspection below shows misclassifications
VOTE_CONFIDENT = 5
VOTE_LIKELY    = 1

conditions = [
    # real_confident: cross-referenced with IMDB and has meaningful votes
    has_imdb & (votes > VOTE_CONFIDENT),

    # real_likely: human engagement or strong cross-reference
    (votes > VOTE_LIKELY) & has_overview,
    has_imdb & has_overview & has_genres,
    votes > 20,

    # stub_legitimate: real but undiscovered (upcoming, foreign, art house)
    (has_imdb | (votes > 0)) & (has_overview | has_genres),

    # stub_uncertain: some data exists but not enough to call
    has_overview | has_genres | has_release,

    # spam_confident: no fields at all
    ~has_imdb & (votes == 0) & ~has_overview & ~has_genres & ~has_release & ~has_runtime,
]
labels = [
    'real_confident',
    'real_likely', 'real_likely', 'real_likely',
    'stub_legitimate',
    'stub_uncertain',
    'spam_confident',
]

df['quality'] = 'spam_likely'  # default
for cond, label in zip(reversed(conditions), reversed(labels)):
    df.loc[cond, 'quality'] = label

ORDER = ['real_confident', 'real_likely', 'stub_legitimate', 'stub_uncertain', 'spam_likely', 'spam_confident']
counts = df['quality'].value_counts().reindex(ORDER)
print(pd.DataFrame({
    'count': counts,
    'pct':   (counts / len(df) * 100).round(1)
}).to_string())

                  count   pct
quality                      
real_confident   108739   9.3
real_likely      325652  27.8
stub_legitimate  193741  16.6
stub_uncertain   469501  40.1
spam_likely       16842   1.4
spam_confident    55281   4.7


## Inspect samples — validate and adjust rules above

In [4]:
COLS = ['id', 'title', 'release_date', 'popularity', 'vote_count', 'genres', 'imdb_id', 'theatrical']

for label in ORDER:
    subset = df[df['quality'] == label].sort_values('vote_count', ascending=False)
    print(f'\n══ {label} ({len(subset):,}) ══')
    print(subset[COLS].head(5).to_string(index=False))
    if len(subset) > 8:
        print('  ...')
        print(subset[COLS].tail(3).to_string(index=False))


══ real_confident (108,739) ══
    id           title release_date  popularity  vote_count                                      genres   imdb_id  theatrical
 27205       Inception   2010-07-15      83.952       34495          Action, Science Fiction, Adventure tt1375666        True
157336    Interstellar   2014-11-05     140.241       32571           Adventure, Drama, Science Fiction tt0816692        True
   155 The Dark Knight   2008-07-16     130.643       30619              Drama, Action, Crime, Thriller tt0468569        True
 19995          Avatar   2009-12-15      79.932       29815 Action, Adventure, Fantasy, Science Fiction tt0499549        True
 24428    The Avengers   2012-04-25      98.082       29166          Science Fiction, Action, Adventure tt0848228        True
  ...
    id                     title release_date  popularity  vote_count         genres   imdb_id  theatrical
199697             Lump of Sugar   2006-08-10       3.900           6 Romance, Drama tt0438986     


══ real_likely (325,652) ══
     id                                         title release_date  popularity  vote_count                    genres imdb_id  theatrical
 567604                          Once Upon a Deadpool   2018-12-11      23.001         646 Comedy, Action, Adventure     NaN        True
 836466                                        Return   2020-06-10       7.915         619                 Animation     NaN        True
1040330             Black Adam: Saviour or Destroyer?   2022-10-15      10.918         404               Documentary     NaN        True
 665399 BTS World Tour: Love Yourself - Japan Edition   2019-10-09       4.892         306        Music, Documentary     NaN       False
1086372                                        Return   1972-07-02       4.835         217                       NaN     NaN        True
  ...
     id       title release_date  popularity  vote_count                 genres    imdb_id  theatrical
 496367   Panchlait   2017-11-17        


══ stub_legitimate (193,741) ══
    id                                      title release_date  popularity  vote_count           genres imdb_id  theatrical
372253                   Artus - Saignant à point   2015-11-10       1.503          19           Comedy     NaN        True
 87182        Eric & Ramzy - Au Palais des Glaces   1998-09-27       1.327          19           Comedy     NaN       False
441883 Jérémy Ferrari - Vends 2 pièces à Beyrouth   2018-03-28       1.555          19           Comedy     NaN        True
572026            Din Don - Una parrocchia in due   2018-01-01       1.646          19 TV Movie, Comedy     NaN        True
784945                                   Identità   2003-12-01       0.637          18      Documentary     NaN        True
  ...
     id         title release_date  popularity  vote_count genres    imdb_id  theatrical
 483927 The Neon Rose   1964-11-13         0.6           0    NaN  tt0365550        True
 483864           Por   2010-02-02     


══ stub_uncertain (469,501) ══
     id                                        title release_date  popularity  vote_count genres imdb_id  theatrical
 597165                                            1   2008-06-01       0.940          18    NaN     NaN        True
1186774 Reunión 10 años – No se aceptan devoluciones   2023-10-03      20.664          17    NaN     NaN        True
 806675                                          XXX   1989-01-01       5.482          13    NaN     NaN        True
 636261                             А теперь суди...   1966-02-02       1.400          12    NaN     NaN        True
 894133                         WWE NXT: TakeOver 36   2021-08-22       0.844          12    NaN     NaN       False
  ...
     id               title release_date  popularity  vote_count      genres imdb_id  theatrical
 274938 Osuofia Papa Africa   2009-01-01         0.6           0         NaN     NaN        True
 274941       WE RIDE TO DC   2014-06-05         0.6           0 D

## Theatrical breakdown per label

Are spam entries mostly video releases?

In [5]:
summary = df.groupby('quality').apply(lambda g: pd.Series({
    'count':           len(g),
    'theatrical_pct':  f"{g['theatrical'].mean()*100:.1f}%",
    'has_keywords_pct':f"{g['keywords'].notna().mean()*100:.1f}%",
    'median_votes':    g['vote_count'].median(),
    'median_pop':      g['popularity'].median(),
})).reindex(ORDER)
print(summary.to_string())

                  count theatrical_pct has_keywords_pct  median_votes  median_pop
quality                                                                          
real_confident   108739          98.6%            66.0%          17.0       2.865
real_likely      325652          97.6%            34.2%           0.0       0.608
stub_legitimate  193741          95.4%            15.8%           0.0       0.600
stub_uncertain   469501          92.6%            14.3%           0.0       0.600
spam_likely       16842          96.3%             4.5%           0.0       0.000
spam_confident    55281          92.6%             3.6%           0.0       0.000


## API spot-check — is Kaggle data stale for low-signal entries?

Sample some `spam_likely` entries and hit the live TMDB API to see if they've
been updated since the Kaggle snapshot.

In [6]:
import asyncio, aiohttp, os
from dotenv import load_dotenv
load_dotenv()

TOKEN   = os.environ['TMDB_TOKEN']
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'accept': 'application/json'}

sample = df[df['quality'] == 'spam_likely'].sample(12, random_state=42)

async def fetch(session, mid):
    async with session.get(
        f'https://api.themoviedb.org/3/movie/{mid}',
        headers=HEADERS, timeout=aiohttp.ClientTimeout(total=10)
    ) as r:
        return mid, r.status, (await r.json() if r.status == 200 else {})

async def main():
    async with aiohttp.ClientSession() as s:
        return await asyncio.gather(*[fetch(s, mid) for mid in sample['id']])

results = await main()

print(f'{"ID":<10} {"Status":<6} {"Votes(API)":>10} {"Votes(Kaggle)":>13}  Title')
print('-' * 80)
for mid, status, data in results:
    row = sample[sample['id'] == mid].iloc[0]
    api_votes = data.get('vote_count', '?')
    api_overview = str(data.get('overview', ''))[:60]
    stale = '← STALE' if isinstance(api_votes, int) and api_votes > row['vote_count'] else ''
    print(f"{mid:<10} {status:<6} {str(api_votes):>10} {str(int(row['vote_count'])):>13}  {row['title'][:35]} {stale}")
    if api_overview:
        print(f"  overview: {api_overview}")

ID         Status Votes(API) Votes(Kaggle)  Title
--------------------------------------------------------------------------------
1643380    200             0             0  Jimmy Flynn's Camp Comedy 
  overview: Jimmy Flynn, this comedy sketch series promises hilarious en
1427199    200             0             0  Dinanzi a noi il cielo 
1641486    200             0             0  Entre dos 
  overview: Two men meet on a park bench and remember a dark episode fro
1089818    200             1             0  Horizon: How Long Is a Piece of Str ← STALE
1210578    200             2             0  La mécanique des choses ← STALE
  overview: At first, there are a wrecked father and a daughter who want
1289769    200             0             0  Maj kompjutr 
  overview: Thesis film about three friends (director, violinist and a s
1130158    200             0             0  Dilmun 
1508248    200             0             0  Love u Hiroshima 
1373528    200             0             0  The

## Export

- `data/tmdb_movies_labeled.csv` — full dataset with `quality` + `theatrical` columns
- `data/tmdb_movies_real.csv` — `real_confident` + `real_likely` only

In [7]:
front = ['quality', 'theatrical']
rest  = [c for c in df.columns if c not in front + ['video']]
out   = df[front + rest]

out.to_csv('data/tmdb_movies_labeled.csv', index=False)
print(f'Full labeled: {len(out):,} → data/tmdb_movies_labeled.csv')

real = out[out['quality'].isin(['real_confident', 'real_likely'])]
real.to_csv('data/tmdb_movies_real.csv', index=False)
print(f'Real subset:  {len(real):,} → data/tmdb_movies_real.csv')
print(f'  theatrical: {real["theatrical"].sum():,} ({real["theatrical"].mean()*100:.1f}%)')
print(f'  with keywords: {real["keywords"].notna().sum():,} ({real["keywords"].notna().mean()*100:.1f}%)')

Full labeled: 1,169,756 → data/tmdb_movies_labeled.csv


Real subset:  434,391 → data/tmdb_movies_real.csv
  theatrical: 424,989 (97.8%)
  with keywords: 183,265 (42.2%)
